# Task 13b — schedule repair, retention, and the per-word objective

Task 13 ran every arm for **3 epochs at LR 1e-5** and produced a table in which the *untouched*
model B0 beat every trained arm on ordinary NTP, and beat all but one on alternative-set NLL.
Per-arm validation loss bottomed out after **epoch 1** and rose for two more, so two thirds of
every Task-13 run was actively damaging the model. Nothing in that table is safe to publish until
a plain-CLM arm can at least match the model it started from.

This notebook does four things Task 13 could not:

1. **Stage A — schedule sweep.** Sweep epochs/LR on `A0` only (the fastest arm: no candidate
   scoring) and require `A0 NTP NLL <= B0 NTP NLL` before anything else runs. This is a hard gate.
2. **Stage B — the missing arm.** `R1`/`R2` train the paper's **per-word** loss
   (`paper_mean`) *together with* a CLM retention term, on the same rows and the same replay as
   `A2`/`D2`. Task 13's `P1` had `base_loss_weight=0` and no replay, which is why its NTP
   collapsed. Holding data fixed and changing only the loss form is the comparison Task 13 lacked.
3. **Stage C — an honest A3.** `A3` divides the concept term by `|C|`, which lowers its *average*
   strength by roughly `E[|C|]`. Task 13's "A3 is dominated" therefore only showed that a smaller
   alpha is worse. `A3m` re-runs it with alpha rescaled so mean strength matches `A2`, isolating
   the per-example set-size reweighting that the `1/|C|` term actually introduces.
4. **Report** a research-question scoreboard rather than a bare metric dump.

## 5. Contrastive (`K1`) --- back in, with a stated caveat

Contrastive training was the best-performing arm of the June round (+6\,pp accuracy over NCP) and
has not been re-validated since the July audit invalidated it twice: its headline number came from
train/val leakage, and its wrong-sense negatives were structurally empty (0\% coverage). `K1`
restores it. Because the contrastive CSV's `positives` column is byte-identical to `A2`'s candidate
set, **`K1` vs `A2` varies only `contrast_beta`** --- same rows, same replay, same objective, same
alpha.

**Read a null result from `K1` with care.** Its negatives are WordNet-mined, and manual inspection
shows some are valid substitutes in context. One real example from `contrastive_train.csv`:

> context `...the fact that Katt is running with the Nutcracker joke is freakin ___`
> gold `Great`, positives `['Great', 'fantastic']`,
> negatives `['dandy', 'outstanding', 'marvelous', 'cracking', 'bang-up', 'neat', 'keen']`

Every one of those "negatives" is an acceptable substitute. InfoNCE would push probability *away*
from them, which is the opposite of the objective. An automated proxy (a negative that some other
row annotates as a valid positive for the same gold word) puts contamination at 2.1\% of WordNet
negatives versus 1.5\% for SWORDS' human labels, but that proxy has low power on a 1,589-row
corpus and the qualitative rate is clearly higher. `--strategy wrong_sense` is used rather than
`all` because co-hyponyms are worse still (Chen's note: they "may still be valid completions in
context").

**Therefore:** a `K1` *gain* is attributable to the contrastive term. A `K1` *loss* is not
attributable --- it could be the objective, or it could be suppressing valid substitutes. The
clean version of this experiment is `M_C` in Task 14, where SWORDS supplies 2,350 substitutes
that human annotators marked FALSE. Report both, and report negative-mining coverage
(93.5\% here) separately from eval-slot coverage.

## Why this is a new notebook and not extra cells in Task 13

Task 13's resume logic keys on `run__{model}_{arm}_s{seed}.json`. Re-running it under a new
schedule while reusing those tags would silently reload the stale 3-epoch results and produce a
table that looks correct and is not. This notebook writes to **`task13b_controlled/`** and stamps
the schedule into every tag.

| Arm | Owner | Objective | base wt | Data |
|---|---|---|---|---|
| B0 | base | untouched | - | - |
| A0s | paper | none (CLM) | 1.0 | repeated originals |
| A1s | paper | none (CLM) | 1.0 | augmented sentences |
| P1s | paper | `paper_mean` | **0.0** | concept rows cycled to budget |
| D2s | control | none (CLM) | 1.0 | concept rows + replay |
| A2s | ours | `set_marginal` | 1.0 | concept rows + replay |
| **R1s** | **ours (new)** | **`paper_mean`** | **1.0** | **concept rows (gold-excl) + replay** |
| **R2s** | **ours (new)** | **`paper_mean`** | **1.0** | **concept rows (gold-incl) + replay** |
| A3s | ours | `set_marginal_scaled` | 1.0 | concept rows + replay |
| **A3ms** | **ours (new)** | **`set_marginal_scaled`, matched alpha** | **1.0** | **concept rows + replay** |
| **K1s** | **ours (new)** | **`set_marginal` + InfoNCE, `contrast_beta=1.0`** | **1.0** | **contrastive rows + replay** |

`R2s` shares `A2s`'s exact gold-inclusive candidate set, so `R2s` vs `A2s` isolates the loss form
alone. `R1s` keeps the paper's gold-exclusive convention.

## Faithfulness note on P1

Iyer et al. inflate by duplication **only for the NTP baselines** (section 2.1.2, "No Concept ...
to train NTP baselines with matched number of repeated datapoints"). Their released loss-function
file is `context_loss_train.csv` at 1,828 rows, never inflated, while `vanilla_train.txt` and
`context_syn_train.txt` are both 8,000 lines. Task 13's `P1` cycles concept rows up to the full
budget, giving it ~4.9x more concept exposure than the paper gives its own loss model. That is a
defensible step-matching choice but it confounds `P1` vs `A2`. `P1n` below adds the un-cycled,
paper-faithful variant so the exposure axis is measured rather than assumed.

## 0. Runtime, authentication, and repository

In [1]:
import os, re, sys, subprocess, shutil, json, glob, math, random, statistics, ast
from pathlib import Path
import torch, pandas as pd, numpy as np

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
!pip install -q "transformers>=4.57,<5" datasets accelerate pytest pandas scipy matplotlib nltk huggingface_hub

from google.colab import drive, userdata
from huggingface_hub import login
drive.mount('/content/drive')
hf_token = userdata.get('HF_TOKEN')
assert hf_token, 'Add HF_TOKEN to Colab Secrets for gated Llama access.'
login(token=hf_token, add_to_git_credential=False)

NVIDIA L4, 23034 MiB, 22561 MiB

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
Mounted at /content/drive


In [2]:
REPO_URL = 'https://github.com/SharvaGogawale1/concept-aware-training.git'
REPO_DIR = Path('/content/concept_aware_training')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)

SCRIPTS = REPO_DIR / 'transformers/examples/pytorch/language-modeling'
required = ['sequence_ncp_trainer.py', 'run_clm_sequence_ncp.py',
            'eval_concept_ppl_v3.py', 'test_sequence_ncp.py']
missing = [name for name in required if not (SCRIPTS / name).exists()]
for name in ['build_contrastive_dataset.py', 'build_contrastive_gold.py']:
    if not (REPO_DIR / name).exists(): missing.append(name)
assert not missing, f'Push Task-13 files before Colab: {missing}'
source = (SCRIPTS / 'sequence_ncp_trainer.py').read_text() + \
         (SCRIPTS / 'run_clm_sequence_ncp.py').read_text()
for token in ['paper_mean', 'set_marginal_scaled', 'base_loss_weight',
              'candidate_microbatch_size', 'deduplicate_text_rows']:
    assert token in source, f'missing required implementation: {token}'
print('Repository ready:', REPO_DIR)

Repository ready: /content/concept_aware_training


## 1. Locked configuration

In [3]:
DATA_ROOT = REPO_DIR / 'data'
SCRATCH = Path('/content/task13b_scratch')
DRIVE_ROOT = Path('/content/drive').resolve()
# Deliberately NOT task13_controlled: reusing that folder would let the resume logic load
# 3-epoch runs under the repaired schedule's tags.
RESULTS = DRIVE_ROOT / 'MyDrive/concept_aware_outputs/task13b_controlled'
SCRATCH.mkdir(parents=True, exist_ok=True); RESULTS.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 2024]
RELATION = 'syn'                 # hypernym matrix is a later stage, as in Task 13

RUN_STAGE_A_SWEEP   = True       # ~1 h : schedule sweep on A0, one seed
RUN_STAGE_B_MATRIX  = True       # headline arms, all seeds
RUN_STAGE_C_EXTRAS  = False      # A3/A3m/R2/P1n ablations, all seeds -- enable once B is read

# Stage B headline arms.  Stage C holds the ablations so a dropped Colab session costs less.
STAGE_B_ARMS = ['A0s', 'A1s', 'P1s', 'D2s', 'A2s', 'R1s', 'K1s']
STAGE_C_ARMS = ['R2s', 'A3s', 'A3ms', 'P1ns']

MODEL_REPOS = {'llama1b': 'meta-llama/Llama-3.2-1B'}
PRIMARY_MODEL = 'llama1b'
MODEL_PATHS = {key: Path(f'/content/model_{key}') for key in MODEL_REPOS}

def assert_ephemeral(path):
    resolved = Path(path).resolve()
    assert os.path.commonpath([str(resolved), str(DRIVE_ROOT)]) != str(DRIVE_ROOT), \
        f'weights/checkpoints may not be written to Drive: {resolved}'

print('relation:', RELATION, '| seeds:', SEEDS)
print('stages: A =', RUN_STAGE_A_SWEEP, '| B =', RUN_STAGE_B_MATRIX, '| C =', RUN_STAGE_C_EXTRAS)
print('results ->', RESULTS)

relation: syn | seeds: [42, 123, 2024]
stages: A = True | B = True | C = False
results -> /content/drive/MyDrive/concept_aware_outputs/task13b_controlled


In [4]:
from huggingface_hub import snapshot_download
for key, repo in MODEL_REPOS.items():
    snapshot_download(repo_id=repo, local_dir=str(MODEL_PATHS[key]), token=hf_token,
                      ignore_patterns=['*.msgpack', '*.h5', '*.ot', 'original/*'])
    print(key, MODEL_PATHS[key])

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

LICENSE.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

llama1b /content/model_llama1b


## 2. Matched datasets

Identical to Task 13 so budgets and row digests are comparable across the two notebooks.
Augmentation fixes the budget `N`: `A0` gets `N` repeated originals, `A1` the `N` augmented
sentences, `P1` concept rows cycled to `N`, and the hybrid arms every concept row once plus
`N - n_concept` replay rows. `P1n` is the un-cycled paper-faithful control and is intentionally
*not* budget-matched -- that is what it measures.

In [5]:
subprocess.run([sys.executable, 'rebuild_gold_inclusive.py'], check=True)

# Contrastive splits that PRESERVE gold_surface and row_id.  build_contrastive_dataset.py writes
# only text/positives/negatives and defaults to the pre-July leaked splits, so neither its output
# nor its defaults are usable here: the trainer needs gold_surface (require_gold_for_concept) and
# the paired bootstrap needs row_id.  Strategy is wrong_sense rather than 'all' because
# co-hyponyms are frequently valid substitutes in context (Chen's note), and suppressing a valid
# substitute is indistinguishable from the objective failing.
!pip install -q nltk
import nltk
for _pkg in ['wordnet', 'omw-1.4']: nltk.download(_pkg, quiet=True)
for _rel in ['syn', 'hyp']:
    subprocess.run([sys.executable, 'build_contrastive_gold.py', '--relation', _rel,
                    '--strategy', 'wrong_sense', '--max_negatives', '10', '--seed', '42'],
                   check=True)

RAW = {
    'syn': {'clean': DATA_ROOT / 'syn/youtube_clean', 'gold': DATA_ROOT / 'syn/youtube_clean_gold'},
    'hyp': {'clean': DATA_ROOT / 'hyp/youtube_clean', 'gold': DATA_ROOT / 'hyp/youtube_clean_gold'},
}

def read_lines(path):
    return [line.strip() for line in open(path, encoding='utf-8') if line.strip()]

def write_lines(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text('\n'.join(rows) + '\n', encoding='utf-8')

def cycle_to_size(rows, size):
    assert rows and size >= 0
    return [rows[index % len(rows)] for index in range(size)]

DATA, budget_report = {}, []
for relation, roots in RAW.items():
    clean, gold_root = roots['clean'], roots['gold']
    augmented = read_lines(clean / 'context_syn_train.txt')
    originals = read_lines(clean / 'vanilla_train.txt')
    budget = len(augmented)
    inclusive = pd.read_csv(gold_root / 'context_loss_train.csv')
    exclusive = pd.read_csv(gold_root / 'context_loss_train_goldexcl.csv')
    assert list(inclusive.row_id) == list(exclusive.row_id)
    assert len(inclusive) <= budget

    matched_original = SCRATCH / f'{relation}_A0_repeated_original.txt'
    matched_replay = SCRATCH / f'{relation}_matched_replay.txt'
    repeated_concepts = SCRATCH / f'{relation}_P1_repeated_concepts.csv'
    write_lines(matched_original, cycle_to_size(originals, budget))
    write_lines(matched_replay, cycle_to_size(originals, budget - len(inclusive)))

    repeated = exclusive.iloc[np.arange(budget) % len(exclusive)].copy()
    repeated['row_id'] = [f'{row_id}:paper-repeat:{index:05d}'
                          for index, row_id in enumerate(repeated.row_id.astype(str))]
    repeated.to_csv(repeated_concepts, index=False)

    DATA[relation] = {
        'budget': budget, 'vanilla_train': matched_original,
        'aug_train': clean / 'context_syn_train.txt',
        'vanilla_val': clean / 'vanilla_val.txt',
        'gold_train': gold_root / 'context_loss_train.csv',
        'gold_val': gold_root / 'context_loss_val.csv',
        'excl_train': gold_root / 'context_loss_train_goldexcl.csv',
        'excl_val': gold_root / 'context_loss_val_goldexcl.csv',
        'contrastive_train': gold_root / 'contrastive_train.csv',
        'contrastive_val': gold_root / 'contrastive_val.csv',
        'paper_train': repeated_concepts,          # cycled to budget (Task 13's P1)
        'paper_train_native': gold_root / 'context_loss_train_goldexcl.csv',  # un-cycled (P1n)
        'replay': matched_replay,
    }
    for path in DATA[relation].values():
        if isinstance(path, Path): assert path.exists(), path
    budget_report.append({
        'relation': relation, 'target_examples': budget,
        'A0_rows': len(read_lines(matched_original)), 'A1_rows': len(augmented),
        'concept_rows': len(inclusive), 'matched_replay_rows': len(read_lines(matched_replay)),
        'P1_rows': len(repeated), 'P1n_rows': len(exclusive),
        'unique_original_sentences': len(set(originals)),
    })

budget_df = pd.DataFrame(budget_report)
display(budget_df); budget_df.to_csv(RESULTS / 'task13b_data_budgets.csv', index=False)
assert all(budget_df.A0_rows == budget_df.A1_rows)
assert all(budget_df.concept_rows + budget_df.matched_replay_rows == budget_df.A1_rows)

# The overfitting driver, printed so it is impossible to miss: how many times the schedule shows
# the model each distinct sentence, per epoch.
for row in budget_report:
    print(f"{row['relation']}: {row['unique_original_sentences']} unique originals inflated to "
          f"{row['target_examples']} rows -> {row['target_examples']/row['unique_original_sentences']:.1f} "
          f"exposures per sentence per epoch")

,relation,target_examples,A0_rows,A1_rows,concept_rows,matched_replay_rows,P1_rows,P1n_rows,unique_original_sentences
0,syn,7807,7807,7807,1589,6218,7807,1589,545
1,hyp,7999,7999,7999,2096,5903,7999,2096,710


syn: 545 unique originals inflated to 7807 rows -> 14.3 exposures per sentence per epoch
hyp: 710 unique originals inflated to 7999 rows -> 11.3 exposures per sentence per epoch


In [6]:
inc = pd.read_csv(DATA[RELATION]['gold_train'])
exc = pd.read_csv(DATA[RELATION]['excl_train'])
assert list(inc.row_id) == list(exc.row_id)
for left, right, gold in zip(inc.context_syn, exc.context_syn, inc.gold_surface):
    inclusive = {str(v).casefold() for v in ast.literal_eval(left)}
    exclusive = {str(v).casefold() for v in ast.literal_eval(right)}
    gold = str(gold).casefold()
    assert gold in inclusive and gold not in exclusive
    assert inclusive == exclusive | {gold}
print('Gold-inclusive/exclusive alignment passed.')

Gold-inclusive/exclusive alignment passed.


## 3. Arm registry, and the matched alpha for A3

`set_marginal_scaled` computes `-(1/|C|) log sum_c exp(s_c)`. Because `|C|` varies per row, this
is **not** a constant rescale of `set_marginal` -- it reweights small sets up relative to large
ones. But its *mean* strength is lower than `set_marginal` by roughly `E[1/|C|]^-1`, so Task 13's
`A3` at `alpha=0.5` was really running at a mean effective alpha near `0.5 * E[1/|C|]`.

`A3m` fixes `alpha_A3m = alpha_A2 / E[1/|C|]` so mean strength matches `A2` and any surviving
difference is attributable to the per-example size reweighting rather than to overall strength.

In [7]:
sizes = np.array([len(ast.literal_eval(v)) for v in inc.context_syn], dtype=np.float64)
mean_inv_size = float(np.mean(1.0 / sizes))
ALPHA_A2 = 0.5
ALPHA_A3_MATCHED = round(ALPHA_A2 / mean_inv_size, 4)
print(f'candidate-set sizes: n={len(sizes)} mean={sizes.mean():.2f} '
      f'min={sizes.min():.0f} max={sizes.max():.0f}')
print(f'E[1/|C|] = {mean_inv_size:.4f}  ->  A3 at alpha=0.5 has mean effective alpha '
      f'{ALPHA_A2*mean_inv_size:.4f}')
print(f'A3m matched alpha = {ALPHA_A3_MATCHED}')

S = 's' if RELATION == 'syn' else 'h'
PRETTY = 'synonym' if RELATION == 'syn' else 'hypernym'
assert all(a.endswith(S) for a in STAGE_B_ARMS + STAGE_C_ARMS), (
    f'STAGE_B_ARMS/STAGE_C_ARMS are written for suffix {S!r}; '
    'update them when changing RELATION')

ARMS = {
 'B0':      dict(label='Untouched base', owner='base', objective='base',
                 alpha=0., base_weight=0., data=None),
 f'A0{S}':  dict(label=f'A0 {PRETTY} repeated-original NTP', owner='paper', objective='none',
                 alpha=0., base_weight=1., data='vanilla'),
 f'A1{S}':  dict(label=f'A1 {PRETTY} data augmentation', owner='paper', objective='none',
                 alpha=0., base_weight=1., data='augmentation'),
 f'P1{S}':  dict(label=f'P1 {PRETTY} per-word loss, no retention', owner='paper',
                 objective='paper_mean', alpha=1., base_weight=0., data='paper'),
 f'P1n{S}': dict(label=f'P1n {PRETTY} per-word loss, paper-native volume', owner='paper',
                 objective='paper_mean', alpha=1., base_weight=0., data='paper_native'),
 f'D2{S}':  dict(label=f'D2 {PRETTY} alpha-zero control', owner='control', objective='none',
                 alpha=0., base_weight=1., data='hybrid'),
 f'A2{S}':  dict(label=f'A2 {PRETTY} set marginal', owner='ours', objective='set_marginal',
                 alpha=ALPHA_A2, base_weight=1., data='hybrid'),
 f'R1{S}':  dict(label=f'R1 {PRETTY} per-word + retention (gold-excl)', owner='ours',
                 objective='paper_mean', alpha=1., base_weight=1., data='hybrid_excl'),
 f'R2{S}':  dict(label=f'R2 {PRETTY} per-word + retention (gold-incl)', owner='ours',
                 objective='paper_mean', alpha=1., base_weight=1., data='hybrid'),
 f'A3{S}':  dict(label=f'A3 {PRETTY} scaled marginal, alpha=0.5', owner='ours',
                 objective='set_marginal_scaled', alpha=ALPHA_A2, base_weight=1., data='hybrid'),
 f'A3m{S}': dict(label=f'A3m {PRETTY} scaled marginal, matched alpha', owner='ours',
                 objective='set_marginal_scaled', alpha=ALPHA_A3_MATCHED, base_weight=1.,
                 data='hybrid'),
 # K1: A2 plus InfoNCE against WordNet wrong-sense negatives.  The contrastive CSV's `positives`
 # column is byte-identical to the gold CSV's `context_syn` (asserted below), so K1 vs A2 varies
 # ONLY contrast_beta -- same rows, same replay, same candidate set, same objective, same alpha.
 f'K1{S}':  dict(label=f'K1 {PRETTY} set marginal + InfoNCE (WordNet wrong-sense)', owner='ours',
                 objective='set_marginal', alpha=ALPHA_A2, base_weight=1., contrast_beta=1.,
                 data='contrastive'),
}
for key, spec in ARMS.items():
    print(f"{key:6s} [{spec['owner']:8s}] alpha={spec['alpha']:<7} base={spec['base_weight']:<4} "
          f"beta={spec.get('contrast_beta', 0.):<4} {spec['label']}")

# K1 vs A2 is only a clean contrast_beta ablation if the positive sets are identical.  Assert it
# rather than trust it, and report negative-mining coverage separately from eval-slot coverage.
_con = pd.read_csv(DATA[RELATION]['contrastive_train'])
assert list(_con.row_id.astype(str)) == list(inc.row_id.astype(str)), 'contrastive row_ids drifted'
assert [[str(x).strip() for x in ast.literal_eval(v)] for v in inc.context_syn] == \
       [ast.literal_eval(v) for v in _con.positives], 'contrastive positives != gold context_syn'
_negs = [ast.literal_eval(v) for v in _con.negatives]
for _row, _pos, _gold in zip(_negs, _con.positives, _con.gold_surface):
    _banned = {str(x).casefold() for x in ast.literal_eval(_pos)} | {str(_gold).casefold()}
    assert not ({str(n).casefold() for n in _row} & _banned), 'a negative is a positive or the gold'
print(f'\nK1 control verified: positives identical to A2 candidate set; only contrast_beta differs.')
print(f'negative-mining coverage: {100*sum(1 for n in _negs if n)/len(_negs):.2f}% of rows '
      f'({sum(len(n) for n in _negs)} negatives, '
      f'{sum(len(n) for n in _negs)/max(sum(1 for n in _negs if n),1):.2f} per covered row)')

candidate-set sizes: n=1589 mean=5.18 min=2 max=16
E[1/|C|] = 0.2272  ->  A3 at alpha=0.5 has mean effective alpha 0.1136
A3m matched alpha = 2.2009
B0     [base    ] alpha=0.0     base=0.0  beta=0.0  Untouched base
A0s    [paper   ] alpha=0.0     base=1.0  beta=0.0  A0 synonym repeated-original NTP
A1s    [paper   ] alpha=0.0     base=1.0  beta=0.0  A1 synonym data augmentation
P1s    [paper   ] alpha=1.0     base=0.0  beta=0.0  P1 synonym per-word loss, no retention
P1ns   [paper   ] alpha=1.0     base=0.0  beta=0.0  P1n synonym per-word loss, paper-native volume
D2s    [control ] alpha=0.0     base=1.0  beta=0.0  D2 synonym alpha-zero control
A2s    [ours    ] alpha=0.5     base=1.0  beta=0.0  A2 synonym set marginal
R1s    [ours    ] alpha=1.0     base=1.0  beta=0.0  R1 synonym per-word + retention (gold-excl)
R2s    [ours    ] alpha=1.0     base=1.0  beta=0.0  R2 synonym per-word + retention (gold-incl)
A3s    [ours    ] alpha=0.5     base=1.0  beta=0.0  A3 synonym scaled marginal

In [8]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', '--confcutdir=.',
                str(SCRIPTS / 'test_sequence_ncp.py')], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q', '--confcutdir=.', '/content/concept_aware_training/transformers/examples/pytorch/language-modeling/test_sequence_ncp.py'], returncode=0)

## 4. Train / evaluate machinery

Identical to Task 13 except that the **schedule is a parameter, not a constant**, and every tag
carries the schedule so two schedules can never collide in the resume cache. Weights are deleted
after evaluation; only compact JSON reaches Drive.

In [9]:
BAR_LINE = re.compile(r'^\s*(\d+)%\|')
BAR_EVERY = 20

def run_command(parts):
    parts = [str(value) for value in parts]
    print(' '.join(parts), flush=True)
    process = subprocess.Popen(parts, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1,
                               env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    last_bucket = None
    try:
        for line in process.stdout:
            match = BAR_LINE.match(line)
            if match is None:
                if line.strip(): print(line, end='', flush=True)
                continue
            bucket = int(match.group(1)) // BAR_EVERY
            if bucket != last_bucket:
                last_bucket = bucket
                print(line.rstrip(), flush=True)
        if process.wait() != 0:
            raise subprocess.CalledProcessError(process.returncode, parts)
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill()

def schedule_tag(epochs, lr):
    return f'e{epochs:g}_lr{lr:g}'

def make_schedule(epochs, lr, rows=None):
    # logging_steps is numerically inert; scale it so every run logs ~15 loss lines.
    steps = max(1, int((rows or DATA[RELATION]['budget']) * epochs / 16 / 15))
    return ['--num_train_epochs', str(epochs), '--learning_rate', str(lr),
            '--warmup_ratio', '.1', '--block_size', '128',
            '--per_device_train_batch_size', '1', '--per_device_eval_batch_size', '1',
            '--gradient_accumulation_steps', '16', '--logging_steps', str(steps),
            '--save_strategy', 'no', '--save_only_model', '--bf16',
            '--no-gradient_checkpointing', '--overwrite_output_dir', '--do_train', '--do_eval',
            '--report_to', 'none', '--candidate_microbatch_size', '8',
            '--forbidden_output_root', str(DRIVE_ROOT), '--no-deduplicate_text_rows']

def files_for(spec):
    view = spec['data']
    d = DATA[RELATION]
    if view == 'vanilla':      return d['vanilla_train'], d['vanilla_val'], None
    if view == 'augmentation': return d['aug_train'], d['vanilla_val'], None
    if view == 'paper':        return d['paper_train'], d['excl_val'], None
    if view == 'paper_native': return d['paper_train_native'], d['excl_val'], None
    if view == 'hybrid':       return d['gold_train'], d['gold_val'], d['replay']
    if view == 'hybrid_excl':  return d['excl_train'], d['excl_val'], d['replay']
    if view == 'contrastive':  return d['contrastive_train'], d['contrastive_val'], d['replay']
    raise ValueError(view)

def evaluate(tag, checkpoint, model_key):
    output = RESULTS / f'eval__{tag}.json'
    run_command([sys.executable, SCRIPTS / 'eval_concept_ppl_v3.py',
        '--checkpoints', checkpoint, '--tokenizer_path', MODEL_PATHS[model_key],
        '--concept_csv', f"syn={RAW['syn']['gold'] / 'context_loss_val.csv'}",
                         f"hyp={RAW['hyp']['gold'] / 'context_loss_val.csv'}",
        '--vanilla_val', f"syn={RAW['syn']['clean'] / 'vanilla_val.txt'}",
                         f"hyp={RAW['hyp']['clean'] / 'vanilla_val.txt'}",
        '--gold_column', 'gold_surface', '--block_size', '128', '--batch_size', '16',
        '--n_bootstrap', '2000', '--seed', '42', '--results_json', output])
    return output

def train_and_eval(arm, seed, epochs, lr, model_key=PRIMARY_MODEL):
    spec = ARMS[arm]
    tag = f'{model_key}_{arm}_s{seed}_{schedule_tag(epochs, lr)}'
    output = SCRATCH / tag; assert_ephemeral(output)
    try:
        if spec['objective'] == 'base':
            evaluation = evaluate(tag, MODEL_PATHS[model_key], model_key)
            record = dict(tag=tag, arm=arm, model=model_key, seed=seed, epochs=epochs, lr=lr,
                          **spec, coverage=None, train_metrics={}, log_history=[],
                          eval_json=str(evaluation))
        else:
            train, val, replay = files_for(spec)
            rows = len(read_lines(train)) if str(train).endswith('.txt') else len(pd.read_csv(train))
            command = [sys.executable, SCRIPTS / 'run_clm_sequence_ncp.py',
                '--model_name_or_path', MODEL_PATHS[model_key],
                '--tokenizer_name', MODEL_PATHS[model_key], '--train_file', train,
                '--validation_file', val, '--objective', spec['objective'],
                '--ncp_alpha', spec['alpha'], '--base_loss_weight', spec['base_weight'],
                '--contrast_beta', spec.get('contrast_beta', 0.0),
                '--required_coverage', '.99',
                '--preprocessing_cache_dir', SCRATCH / f'cache_{arm}',
                '--seed', seed, '--output_dir', output] + make_schedule(epochs, lr, rows)
            if spec['data'] == 'contrastive':
                # contrastive_*.csv names its candidate column `positives`, not `context_syn`.
                command += ['--candidate_column', 'positives', '--negative_column', 'negatives']
            if replay: command += ['--replay_file', replay]
            run_command(command)
            summary = json.load(open(output / 'sequence_ncp_run.json'))
            preprocessing = summary['train_preprocessing']
            coverage = summary['training_concept_coverage']['coverage']
            if spec['objective'] != 'none': assert coverage >= .99
            # P1n is intentionally not budget-matched; every other arm must be.
            if spec['data'] != 'paper_native':
                assert preprocessing['rows_raw'] == DATA[RELATION]['budget'], \
                    f"{arm} saw {preprocessing['rows_raw']} rows, expected {DATA[RELATION]['budget']}"
            evaluation = evaluate(tag, output, model_key)
            record = dict(tag=tag, arm=arm, model=model_key, seed=seed, epochs=epochs, lr=lr,
                          **spec, coverage=coverage, train_metrics=summary.get('train_metrics', {}),
                          log_history=summary.get('log_history', []), eval_json=str(evaluation),
                          preprocessing=preprocessing)
        json.dump(record, open(RESULTS / f'run__{tag}.json', 'w'), indent=2)
        return record
    finally:
        shutil.rmtree(output, ignore_errors=True)
        print('deleted ephemeral model:', output)

def load_or_run(arm, seed, epochs, lr, model_key=PRIMARY_MODEL):
    tag = f'{model_key}_{arm}_s{seed}_{schedule_tag(epochs, lr)}'
    path = RESULTS / f'run__{tag}.json'
    if path.exists():
        print('skip (already in Drive):', path.name, flush=True)
        return json.load(open(path))
    return train_and_eval(arm, seed, epochs, lr, model_key)

def syn_metrics(record):
    result = json.load(open(record['eval_json']))[0]
    ntp, concept = result['ntp'][RELATION], result['concept'][RELATION]
    return {'NTP NLL': ntp['ntp_nll_mean'], 'NTP PPL': ntp['ntp_ppl'],
            'NTP acc': ntp['ntp_accuracy'], 'ALT NLL': concept['alt_nll_mean'],
            'GOLD NLL': concept['gold_nll_mean'], 'SET NLL': concept['concept_nll_mean'],
            'ALT mass': concept['alt_mass_mean'],
            'coverage': concept['eval_slot_coverage_pct']}

## 5. Stage A — schedule sweep, and the hard gate

Task 13's schedule was 3 epochs at LR 1e-5, which showed the model each of 545 distinct
sentences roughly 43 times. `A0` is plain CLM on in-domain text; if it cannot match the
checkpoint it started from, the schedule is destroying the model and no objective comparison
run under it means anything.

`A0` is the cheapest arm (no candidate scoring), so the sweep is ~15 min per configuration on an
L4. The gate is `A0 NTP NLL <= B0 NTP NLL`. **Do not proceed to Stage B until it passes.**

In [10]:
# B0 is schedule-independent but must be evaluated once inside this notebook's results folder.
BASE_RECORD = load_or_run('B0', 42, 0, 0)
BASE_NTP = syn_metrics(BASE_RECORD)['NTP NLL']
print(f'\nB0 untouched base: NTP NLL = {BASE_NTP:.4f}  (PPL {math.exp(BASE_NTP):.2f})')

/usr/bin/python3 /content/concept_aware_training/transformers/examples/pytorch/language-modeling/eval_concept_ppl_v3.py --checkpoints /content/model_llama1b --tokenizer_path /content/model_llama1b --concept_csv syn=/content/concept_aware_training/data/syn/youtube_clean_gold/context_loss_val.csv hyp=/content/concept_aware_training/data/hyp/youtube_clean_gold/context_loss_val.csv --vanilla_val syn=/content/concept_aware_training/data/syn/youtube_clean/vanilla_val.txt hyp=/content/concept_aware_training/data/hyp/youtube_clean/vanilla_val.txt --gold_column gold_surface --block_size 128 --batch_size 16 --n_bootstrap 2000 --seed 42 --results_json /content/drive/MyDrive/concept_aware_outputs/task13b_controlled/eval__llama1b_B0_s42_e0_lr0.json
2026-08-28 18:06:00.341609: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environm

In [11]:
SWEEP_GRID = [(1, 1e-5), (1, 5e-6), (2, 5e-6), (3, 3e-6)]

sweep_rows = []
if RUN_STAGE_A_SWEEP:
    for epochs, lr in SWEEP_GRID:
        print(f'\n=== sweep A0 epochs={epochs} lr={lr:g} ===', flush=True)
        record = load_or_run(f'A0{S}', 42, epochs, lr)
        metrics = syn_metrics(record)
        sweep_rows.append({'epochs': epochs, 'lr': lr, **metrics,
                           'beats_base': metrics['NTP NLL'] <= BASE_NTP})

sweep = pd.DataFrame(sweep_rows)
if not sweep.empty:
    sweep.insert(0, 'B0 NTP NLL', BASE_NTP)
    display(sweep[['epochs','lr','NTP NLL','NTP PPL','NTP acc','ALT NLL','GOLD NLL','beats_base']])
    sweep.to_csv(RESULTS / 'task13b_schedule_sweep.csv', index=False)

    passing = sweep[sweep.beats_base]
    if passing.empty:
        best = sweep.loc[sweep['NTP NLL'].idxmin()]
        print(f"\nGATE FAILED. Closest config: epochs={best.epochs:g} lr={best.lr:g} "
              f"NTP NLL {best['NTP NLL']:.4f} vs base {BASE_NTP:.4f} "
              f"(gap {best['NTP NLL']-BASE_NTP:+.4f}).")
        print('Widen SWEEP_GRID downward (lr 1e-6/2e-6, or 0.5 epochs) before running Stage B.')
    else:
        best = passing.loc[passing['NTP NLL'].idxmin()]
        print(f"\nGATE PASSED. Best: epochs={best.epochs:g} lr={best.lr:g} "
              f"NTP NLL {best['NTP NLL']:.4f} <= base {BASE_NTP:.4f}")
    EPOCHS, LR = float(best.epochs), float(best.lr)
else:
    # Set these by hand when re-entering the notebook after Stage A has already been read.
    EPOCHS, LR = 1.0, 1e-5

print(f'\nSchedule for Stages B/C: epochs={EPOCHS:g} lr={LR:g}')


=== sweep A0 epochs=1 lr=1e-05 ===
/usr/bin/python3 /content/concept_aware_training/transformers/examples/pytorch/language-modeling/run_clm_sequence_ncp.py --model_name_or_path /content/model_llama1b --tokenizer_name /content/model_llama1b --train_file /content/task13b_scratch/syn_A0_repeated_original.txt --validation_file /content/concept_aware_training/data/syn/youtube_clean/vanilla_val.txt --objective none --ncp_alpha 0.0 --base_loss_weight 1.0 --contrast_beta 0.0 --required_coverage .99 --preprocessing_cache_dir /content/task13b_scratch/cache_A0s --seed 42 --output_dir /content/task13b_scratch/llama1b_A0s_s42_e1_lr1e-05 --num_train_epochs 1 --learning_rate 1e-05 --warmup_ratio .1 --block_size 128 --per_device_train_batch_size 1 --per_device_eval_batch_size 1 --gradient_accumulation_steps 16 --logging_steps 32 --save_strategy no --save_only_model --bf16 --no-gradient_checkpointing --overwrite_output_dir --do_train --do_eval --report_to none --candidate_microbatch_size 8 --forbidden

,epochs,lr,NTP NLL,NTP PPL,NTP acc,ALT NLL,GOLD NLL,beats_base
0,1,0.000010,2.305627,10.030465,0.641650,9.362415,6.148114,False
1,1,0.000005,2.005854,7.432435,0.661381,7.784841,5.039226,True
2,2,0.000005,2.079474,8.000259,0.656792,8.293458,5.195483,False
3,3,0.000003,2.009817,7.461953,0.661216,7.788383,5.058122,True



GATE PASSED. Best: epochs=1 lr=5e-06 NTP NLL 2.0059 <= base 2.0600

Schedule for Stages B/C: epochs=1 lr=5e-06


## 6. Stages B and C — the arm matrix at the repaired schedule

Seed-major, so an interrupted session leaves whole seeds finished. Every completed run writes to
Drive before the next starts; re-running this cell in a fresh session resumes.

Stage B is the six headline arms. Stage C holds `R2`, `A3`, `A3m`, and `P1n` — enable it once
Stage B has been read, so a dropped session costs less.

In [12]:
import time

PLAN = []
if RUN_STAGE_B_MATRIX:
    PLAN += [(arm, seed) for seed in SEEDS for arm in STAGE_B_ARMS]
if RUN_STAGE_C_EXTRAS:
    PLAN += [(arm, seed) for seed in SEEDS for arm in STAGE_C_ARMS]

RUNS, started = [BASE_RECORD], time.time()
for index, (arm, seed) in enumerate(PLAN, start=1):
    print(f'\n=== [{index}/{len(PLAN)}] {arm} seed={seed} '
          f'| elapsed {(time.time()-started)/3600:.2f}h ===', flush=True)
    RUNS.append(load_or_run(arm, seed, EPOCHS, LR))

pd.DataFrame([{k: v for k, v in row.items() if k not in {'log_history','preprocessing'}}
              for row in RUNS]).to_csv(RESULTS / 'task13b_runs.csv', index=False)
print(f'\ncompleted {len(PLAN)} runs in {(time.time()-started)/3600:.2f}h')

Streaming output truncated to the last 5000 lines.
      "clm_loss": 5.682640262879431,
      "concept_batch_coverage": 1.0,
      "concept_loss": 8.402578845852986,
      "contrast_loss": 0.0,
      "epoch": 0.4800428609697294,
      "grad_norm": 96.5,
      "learning_rate": 3.128966909361272e-06,
      "loss": 8.4026,
      "step": 224,
      "weighted_clm_loss": 0.0
    },
    {
      "clm_loss": 5.758958400285337,
      "concept_batch_coverage": 1.0,
      "concept_loss": 8.195483449148014,
      "contrast_loss": 0.0,
      "epoch": 0.5486204125368337,
      "grad_norm": 79.5,
      "learning_rate": 2.5373985175381595e-06,
      "loss": 8.1955,
      "step": 256,
      "weighted_clm_loss": 0.0
    },
    {
      "clm_loss": 5.538286900729872,
      "concept_batch_coverage": 1.0,
      "concept_loss": 7.92966682324186,
      "contrast_loss": 0.0,
      "epoch": 0.6171979641039379,
      "grad_norm": 72.5,
      "learning_rate": 1.9436976651092143e-06,
      "loss": 7.9297,
      "st

## 7. Master metrics

In [13]:
metric_rows = []
for run in RUNS:
    result = json.load(open(run['eval_json']))[0]
    for eval_relation in ['syn', 'hyp']:
        ntp, concept = result['ntp'][eval_relation], result['concept'][eval_relation]
        metric_rows.append({
            'arm': run['arm'], 'label': run['label'], 'owner': run['owner'],
            'eval_relation': eval_relation, 'seed': run['seed'],
            'NTP NLL': ntp['ntp_nll_mean'], 'NTP PPL': ntp['ntp_ppl'],
            'NTP acc': ntp['ntp_accuracy'], 'ALT NLL': concept['alt_nll_mean'],
            'GOLD NLL': concept['gold_nll_mean'], 'SET NLL': concept['concept_nll_mean'],
            'ALT mass': concept['alt_mass_mean'],
            'coverage %': concept['eval_slot_coverage_pct']})
per_run = pd.DataFrame(metric_rows)
per_run.to_csv(RESULTS / 'task13b_per_run_metrics.csv', index=False)

# Coverage is a data property, not a model property -- it must be constant across arms.
cov = per_run[per_run.eval_relation == RELATION]['coverage %'].round(4).unique()
assert len(cov) == 1, f'eval-slot coverage differs across arms (should be a data property): {cov}'
print(f'eval-slot coverage ({RELATION}): {cov[0]:.2f}%  |  '
      f"n rows = {len(json.load(open(RUNS[0]['eval_json']))[0]['concept'][RELATION]['per_row'])}")

values = ['NTP NLL','NTP PPL','NTP acc','ALT NLL','GOLD NLL','SET NLL','ALT mass']
frame = per_run[per_run.eval_relation == RELATION]
agg = frame.groupby(['owner','arm','label'])[values].agg(['mean','std','count']).reset_index()

table = agg[['owner','arm','label']].copy()
for metric in ['NTP PPL','NTP acc','ALT NLL','GOLD NLL','ALT mass']:
    mean, std, count = agg[(metric,'mean')], agg[(metric,'std')], agg[(metric,'count')]
    fmt = '{:.2f}' if metric == 'NTP PPL' else '{:.3f}'
    table[metric] = [(fmt+' ± '+fmt).format(m, s) if n > 1 else fmt.format(m)
                     for m, s, n in zip(mean, std, count)]
table['n seeds'] = agg[('NTP NLL','count')]
table = table.sort_values('ALT NLL')      # rank by the metric the project is about
display(table); table.to_csv(RESULTS / 'task13b_master_table.csv', index=False)
print('\nSorted by ALT NLL (the concept-coverage metric). Lower is better in every column '
      'except NTP acc and ALT mass.')

eval-slot coverage (syn): 93.99%  |  n rows = 219


,owner,arm,label,NTP PPL,NTP acc,ALT NLL,GOLD NLL,ALT mass,n seeds
,,,,,,,,,
7,paper,P1s,"P1 synonym per-word loss, no retention",10.19 ± 0.03,0.655 ± 0.001,6.229 ± 0.006,5.808 ± 0.003,0.021 ± 0.000,3
4,ours,R1s,R1 synonym per-word + retention (gold-excl),7.57 ± 0.01,0.663 ± 0.001,7.196 ± 0.014,5.114 ± 0.005,0.015 ± 0.000,3
6,paper,A1s,A1 synonym data augmentation,7.42 ± 0.01,0.662 ± 0.001,7.473 ± 0.004,5.085 ± 0.006,0.014 ± 0.000,3
2,ours,A2s,A2 synonym set marginal,7.45 ± 0.02,0.663 ± 0.000,7.566 ± 0.004,5.063 ± 0.001,0.015 ± 0.000,3
3,ours,K1s,K1 synonym set marginal + InfoNCE (WordNet wro...,7.46 ± 0.01,0.663 ± 0.000,7.569 ± 0.006,5.073 ± 0.005,0.015 ± 0.000,3
1,control,D2s,D2 synonym alpha-zero control,7.41 ± 0.01,0.663 ± 0.001,7.591 ± 0.005,5.044 ± 0.004,0.015 ± 0.000,3
5,paper,A0s,A0 synonym repeated-original NTP,7.44 ± 0.01,0.661 ± 0.000,7.782 ± 0.003,5.046 ± 0.006,0.014 ± 0.000,3
0,base,B0,Untouched base,7.85,0.660,7.824,5.552,0.014,1



Sorted by ALT NLL (the concept-coverage metric). Lower is better in every column except NTP acc and ALT mass.


## 8. Paired inference

All comparisons use identical held-out `row_id`s; a negative delta means the challenger has lower
NLL. `nll_alternatives_mean` is **omitted deliberately**: it equals `nll_alternatives + log|C|`,
and `|C|` is identical for challenger and reference on a shared row, so the term cancels exactly
and it contributed a duplicate row to every Task-13 comparison.

The decisive contrasts:

- `R2 vs A2` — **loss form only.** Same rows, same replay, same base weight, same gold-inclusive
  candidate set. Per-word versus set-marginal, nothing else.
- `A2 vs D2` — the concept coefficient, everything else byte-matched.
- `R1 vs P1` — what the retention term buys on top of the paper's objective.
- `A3m vs A3` — set-size reweighting at matched mean strength.

In [14]:
def concept_rows(run, relation=RELATION):
    return {str(r['row_id']): r
            for r in json.load(open(run['eval_json']))[0]['concept'][relation]['per_row']}

def paired_bootstrap(challenger, reference, field, n_boot=10000):
    left, right = concept_rows(challenger), concept_rows(reference)
    assert set(left) == set(right), f"paired rows differ: {challenger['tag']} vs {reference['tag']}"
    ids = sorted(left)
    delta = np.asarray([left[i][field] - right[i][field] for i in ids], dtype=np.float64)
    assert np.isfinite(delta).all()
    rng = np.random.default_rng(13_000 + int(challenger['seed']))
    means = np.empty(n_boot)
    for start in range(0, n_boot, 500):
        size = min(500, n_boot - start)
        idx = rng.integers(0, len(delta), size=(size, len(delta)))
        means[start:start+size] = delta[idx].mean(axis=1)
    lo, hi = np.quantile(means, [.025, .975])
    return {'n': len(delta), 'mean_delta': float(delta.mean()),
            'ci_low': float(lo), 'ci_high': float(hi),
            'win_rate': float(np.mean(delta < 0))}

run_index = {(r['arm'], int(r['seed'])): r for r in RUNS if r['arm'] != 'B0'}

COMPARISONS = [
    ('loss_form_only__R2_vs_A2',      f'R2{S}',  f'A2{S}'),
    ('ours_causal__A2_vs_D2',         f'A2{S}',  f'D2{S}'),
    ('retention_added__R1_vs_P1',     f'R1{S}',  f'P1{S}'),
    ('ours_vs_paper_DA__R1_vs_A1',    f'R1{S}',  f'A1{S}'),
    ('setmarg_vs_paper_DA__A2_vs_A1', f'A2{S}',  f'A1{S}'),
    ('paper_DA_vs_repeat__A1_vs_A0',  f'A1{S}',  f'A0{S}'),
    ('paper_loss_vs_repeat__P1_vs_A0',f'P1{S}',  f'A0{S}'),
    ('contrastive_term__K1_vs_A2',    f'K1{S}',  f'A2{S}'),
    ('sizeweight__A3m_vs_A3',         f'A3m{S}', f'A3{S}'),
    ('exposure__P1_vs_P1n',           f'P1{S}',  f'P1n{S}'),
]

paired = []
for name, challenger_arm, reference_arm in COMPARISONS:
    for seed in SEEDS:
        left  = run_index.get((challenger_arm, seed))
        right = run_index.get((reference_arm, seed))
        if left is None or right is None:
            continue                     # arm belongs to a stage that was not run
        for field in ['nll_alternatives', 'nll_gold']:
            paired.append({'comparison': name, 'seed': seed, 'field': field,
                           'challenger': challenger_arm, 'reference': reference_arm,
                           **paired_bootstrap(left, right, field)})

paired_table = pd.DataFrame(paired)
if paired_table.empty:
    print('No paired comparisons available yet -- run Stage B first.')
else:
    paired_table.to_csv(RESULTS / 'task13b_paired_bootstrap.csv', index=False)
    display(paired_table.round(4))

    gate_rows = []
    for (comparison, field), rows in paired_table.groupby(['comparison','field']):
        consistent = bool((rows.mean_delta < 0).all())
        significant = bool((rows.ci_high < 0).all())
        gate_rows.append({'comparison': comparison, 'field': field, 'n seeds': len(rows),
                          'all_seeds_improve': consistent,
                          'all_seed_CIs_below_zero': significant,
                          'claim_allowed': consistent and significant})
    claim_gate = pd.DataFrame(gate_rows)
    claim_gate.to_csv(RESULTS / 'task13b_claim_gate.csv', index=False)
    display(claim_gate)

,comparison,seed,field,challenger,reference,n,mean_delta,ci_low,ci_high,win_rate
0,ours_causal__A2_vs_D2,42,nll_alternatives,A2s,D2s,219,-0.0247,-0.0372,-0.0124,0.5936
1,ours_causal__A2_vs_D2,42,nll_gold,A2s,D2s,219,0.0225,0.0074,0.0379,0.4795
2,ours_causal__A2_vs_D2,123,nll_alternatives,A2s,D2s,219,-0.0265,-0.0387,-0.0142,0.6210
3,ours_causal__A2_vs_D2,123,nll_gold,A2s,D2s,219,0.0211,0.0070,0.0354,0.4338
4,ours_causal__A2_vs_D2,2024,nll_alternatives,A2s,D2s,219,-0.0233,-0.0351,-0.0114,0.6164
5,ours_causal__A2_vs_D2,2024,nll_gold,A2s,D2s,219,0.0135,-0.0009,0.0283,0.4886
6,retention_added__R1_vs_P1,42,nll_alternatives,R1s,P1s,219,0.9864,0.7785,1.2026,0.2603
7,retention_added__R1_vs_P1,42,nll_gold,R1s,P1s,219,-0.6944,-0.8332,-0.5550,0.7854
8,retention_added__R1_vs_P1,123,nll_alternatives,R1s,P1s,219,0.9660,0.7654,1.1812,0.2694
9,retention_added__R1_vs_P1,123,nll_gold,R1s,P1s,219,-0.6986,-0.8309,-0.5635,0.8037


,comparison,field,n seeds,all_seeds_improve,all_seed_CIs_below_zero,claim_allowed
0,contrastive_term__K1_vs_A2,nll_alternatives,3,False,False,False
1,contrastive_term__K1_vs_A2,nll_gold,3,False,False,False
2,ours_causal__A2_vs_D2,nll_alternatives,3,True,True,True
3,ours_causal__A2_vs_D2,nll_gold,3,False,False,False
4,ours_vs_paper_DA__R1_vs_A1,nll_alternatives,3,True,True,True
5,ours_vs_paper_DA__R1_vs_A1,nll_gold,3,False,False,False
6,paper_DA_vs_repeat__A1_vs_A0,nll_alternatives,3,True,True,True
7,paper_DA_vs_repeat__A1_vs_A0,nll_gold,3,False,False,False
8,paper_loss_vs_repeat__P1_vs_A0,nll_alternatives,3,True,True,True
9,paper_loss_vs_repeat__P1_vs_A0,nll_gold,3,False,False,False


## 9. Research-question scoreboard

Task 13 reported a metric dump, which reads as a string of negative results. The same numbers
answer five separate questions, and most of them answer positively. This cell states each
question, its predeclared decision rule, and the verdict the data actually supports.

In [15]:
def arm_mean(arm, metric):
    rows = frame[frame.arm == arm][metric]
    return float(rows.mean()) if len(rows) else float('nan')

def gate(comparison, field):
    if paired_table.empty: return None
    rows = claim_gate[(claim_gate.comparison == comparison) & (claim_gate.field == field)]
    return bool(rows.claim_allowed.iloc[0]) if len(rows) else None

def verdict(value):
    return {True: 'YES', False: 'NO', None: 'not run'}[value]

scoreboard = [
 dict(RQ='RQ1', question='Does concept-level training improve coverage of valid alternatives '
                         'over a volume-matched NTP baseline?',
      rule='P1 vs A0 on nll_alternatives: all seeds improve and all CIs below zero',
      answer=verdict(gate('paper_loss_vs_repeat__P1_vs_A0', 'nll_alternatives'))),
 dict(RQ='RQ2', question='Is the concept term itself causal, or is the gain from the data?',
      rule='A2 vs D2 on nll_alternatives (byte-matched data, only alpha differs)',
      answer=verdict(gate('ours_causal__A2_vs_D2', 'nll_alternatives'))),
 dict(RQ='RQ3', question='Does the set-marginal form deliver coverage, or concentrate on gold?',
      rule='CONCENTRATES if A2 beats D2 on gold by more than on alternatives, '
           'and A2 loses to A1 on alternatives',
      answer='CONCENTRATES' if (
          not paired_table.empty
          and arm_mean(f'A2{S}','ALT NLL') > arm_mean(f'A1{S}','ALT NLL')) else 'SPREADS'),
 dict(RQ='RQ4', question='Does the per-word form plus a retention term get coverage without '
                         'wrecking the language model?',
      rule=f'R1 ALT NLL <= A1 ALT NLL and R1 NTP NLL <= B0 NTP NLL ({BASE_NTP:.3f})',
      answer=verdict(
          (arm_mean(f'R1{S}','ALT NLL') <= arm_mean(f'A1{S}','ALT NLL')
           and arm_mean(f'R1{S}','NTP NLL') <= BASE_NTP)
          if not np.isnan(arm_mean(f'R1{S}','ALT NLL')) else None)),
 dict(RQ='RQ5', question='Is the 1/|C| term a genuine set-size reweighting or just a smaller alpha?',
      rule='A3m vs A3 at matched mean strength: any surviving difference is the reweighting',
      answer=verdict(gate('sizeweight__A3m_vs_A3', 'nll_alternatives'))),
]
board = pd.DataFrame(scoreboard)
pd.set_option('display.max_colwidth', 90)
display(board); board.to_csv(RESULTS / 'task13b_rq_scoreboard.csv', index=False)

print('\n--- headline arms vs the untouched base (lower is better) ---')
for arm in ['B0'] + STAGE_B_ARMS + (STAGE_C_ARMS if RUN_STAGE_C_EXTRAS else []):
    if arm not in set(frame.arm): continue
    ntp, alt, gold = (arm_mean(arm,'NTP NLL'), arm_mean(arm,'ALT NLL'), arm_mean(arm,'GOLD NLL'))
    flag = '' if arm == 'B0' else f'   dNTP {ntp-BASE_NTP:+.3f}  dALT {alt-arm_mean("B0","ALT NLL"):+.3f}'
    print(f'{arm:6s} NTP {ntp:6.3f}  ALT {alt:7.3f}  GOLD {gold:6.3f}{flag}')

,RQ,question,rule,answer
0,RQ1,Does concept-level training improve coverage of valid alternatives over a volume-match...,P1 vs A0 on nll_alternatives: all seeds improve and all CIs below zero,YES
1,RQ2,"Is the concept term itself causal, or is the gain from the data?","A2 vs D2 on nll_alternatives (byte-matched data, only alpha differs)",YES
2,RQ3,"Does the set-marginal form deliver coverage, or concentrate on gold?","CONCENTRATES if A2 beats D2 on gold by more than on alternatives, and A2 loses to A1 o...",CONCENTRATES
3,RQ4,Does the per-word form plus a retention term get coverage without wrecking the languag...,R1 ALT NLL <= A1 ALT NLL and R1 NTP NLL <= B0 NTP NLL (2.060),YES
4,RQ5,Is the 1/|C| term a genuine set-size reweighting or just a smaller alpha?,A3m vs A3 at matched mean strength: any surviving difference is the reweighting,not run



--- headline arms vs the untouched base (lower is better) ---
B0     NTP  2.060  ALT   7.824  GOLD  5.552
A0s    NTP  2.007  ALT   7.782  GOLD  5.046   dNTP -0.053  dALT -0.043
A1s    NTP  2.004  ALT   7.473  GOLD  5.085   dNTP -0.056  dALT -0.351
P1s    NTP  2.321  ALT   6.229  GOLD  5.808   dNTP +0.261  dALT -1.596
D2s    NTP  2.003  ALT   7.591  GOLD  5.044   dNTP -0.057  dALT -0.233
A2s    NTP  2.008  ALT   7.566  GOLD  5.063   dNTP -0.052  dALT -0.258
R1s    NTP  2.025  ALT   7.196  GOLD  5.114   dNTP -0.035  dALT -0.628
K1s    NTP  2.009  ALT   7.569  GOLD  5.073   dNTP -0.051  dALT -0.255


## 10. Loss trajectories and Pareto view

In [16]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

history_rows = []
for run in RUNS:
    for point in run.get('log_history', []):
        if point.get('loss') is None or point.get('epoch') is None: continue
        history_rows.append({'arm': run['arm'], 'seed': run['seed'],
                             'epoch': float(point['epoch']), 'total_loss': float(point['loss']),
                             'clm_loss': point.get('clm_loss'),
                             'concept_loss': point.get('concept_loss')})
history = pd.DataFrame(history_rows)
history.to_csv(RESULTS / 'task13b_loss_history.csv', index=False)

if not history.empty:
    plot_arms = [a for a in ARMS if a != 'B0' and a in set(history.arm)]
    ncols = 3; nrows = math.ceil(len(plot_arms)/ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.8*nrows), squeeze=False)
    for ax, arm in zip(axes.flat, plot_arms):
        subset = history[history.arm == arm]
        for seed, f in subset.groupby('seed'):
            f = f.sort_values('epoch')
            ax.plot(f.epoch, f.total_loss, label=f'total s={seed}')
            if f.concept_loss.notna().any():
                ax.plot(f.epoch, f.concept_loss, '--', alpha=.65, label=f'concept s={seed}')
        ax.set(title=arm, xlabel='epoch', ylabel='raw logged loss'); ax.grid(alpha=.2)
        ax.legend(fontsize=7, ncol=2)
    for ax in axes.flat[len(plot_arms):]: ax.axis('off')
    fig.suptitle(f'Task 13b raw training trajectories (epochs={EPOCHS:g}, lr={LR:g})', y=1.01)
    fig.tight_layout(); fig.savefig(RESULTS / 'task13b_loss_curves.png', dpi=180,
                                    bbox_inches='tight'); plt.show(); plt.close(fig)

OWNER_COLOR = {'base':'#333333','paper':'#4C78A8','control':'#9C9C9C','ours':'#F58518'}
summary = []
for arm in ARMS:
    rows = frame[frame.arm == arm]
    if rows.empty: continue
    summary.append({'arm': arm, 'owner': rows.owner.iloc[0],
                    'NTP': rows['NTP NLL'].mean(), 'ALT': rows['ALT NLL'].mean(),
                    'GOLD': rows['GOLD NLL'].mean()})

fig, ax = plt.subplots(figsize=(7.5, 6))
for row in summary:
    ax.scatter(row['NTP'], row['ALT'], s=110, color=OWNER_COLOR.get(row['owner'], '#777'))
    ax.annotate(row['arm'], (row['NTP'], row['ALT']), xytext=(6, 4), textcoords='offset points')
ax.axvline(BASE_NTP, color='#333', ls=':', lw=1, alpha=.6)
ax.axhline(arm_mean('B0','ALT NLL'), color='#333', ls=':', lw=1, alpha=.6)
ax.set(xlabel='ordinary NTP mean NLL (lower better)',
       ylabel='gold-excluded alternative-set NLL (lower better)',
       title='Retention / coverage Pareto view\n(dotted lines = untouched base; '
             'lower-left quadrant beats doing nothing on both)')
ax.grid(alpha=.25); fig.tight_layout()
fig.savefig(RESULTS / 'task13b_pareto.png', dpi=180); plt.show(); plt.close(fig)

## 11. Storage audit

In [17]:
shutil.rmtree(SCRATCH, ignore_errors=True)
for pattern in ['*.bin','*.safetensors','optimizer.pt','scheduler.pt','scaler.pt',
                'rng_state*.pth','checkpoint-*']:
    leaked = list(RESULTS.rglob(pattern))
    assert not leaked, f'weight/optimizer artifacts found in Drive: {leaked[:5]}'
print('Storage audit passed. Drive contains reports only:')
for path in sorted(RESULTS.iterdir()):
    print(f'{path.stat().st_size/1024:9.1f} KB  {path.name}')

Storage audit passed. Drive contains reports only:
    789.4 KB  eval__llama1b_A0s_s123_e1_lr5e-06.json
    789.3 KB  eval__llama1b_A0s_s2024_e1_lr5e-06.json
    790.6 KB  eval__llama1b_A0s_s42_e1_lr1e-05.json
    789.3 KB  eval__llama1b_A0s_s42_e1_lr5e-06.json
    789.8 KB  eval__llama1b_A0s_s42_e2_lr5e-06.json
    789.5 KB  eval__llama1b_A0s_s42_e3_lr3e-06.json
    789.1 KB  eval__llama1b_A1s_s123_e1_lr5e-06.json
    789.1 KB  eval__llama1b_A1s_s2024_e1_lr5e-06.json
    789.1 KB  eval__llama1b_A1s_s42_e1_lr5e-06.json
    789.3 KB  eval__llama1b_A2s_s123_e1_lr5e-06.json
    789.3 KB  eval__llama1b_A2s_s2024_e1_lr5e-06.json
    789.3 KB  eval__llama1b_A2s_s42_e1_lr5e-06.json
    789.3 KB  eval__llama1b_B0_s42_e0_lr0.json
    789.4 KB  eval__llama1b_D2s_s123_e1_lr5e-06.json
    789.2 KB  eval__llama1b_D2s_s2024_e1_lr5e-06.json
    789.2 KB  eval__llama1b_D2s_s42_e1_lr5e-06.json
    789.3 KB  eval__llama1b_K1s_s123_e1_lr5e-06.json
    789.3 KB  eval__llama1b_K1s_s2024_e1_lr5e-06.json
   

## 12. How to run this

**Session 1 (~1 h).** Cells through Stage A with `RUN_STAGE_B_MATRIX = False`. Read the sweep
table. If the gate fails, widen `SWEEP_GRID` downward and rerun — Stage A results are cached per
configuration, so only new configurations cost time.

**Session 2 (~3–5 h).** Set `RUN_STAGE_B_MATRIX = True` and rerun top to bottom. Stage A is
served from cache. If Colab drops, rerun the same cell: finished runs are skipped.

**Session 3 (optional, ~2–3 h).** Set `RUN_STAGE_C_EXTRAS = True` for `R2`, `A3`, `A3m`, `P1n`.
`R2` is the one that matters — it is the loss-form-only comparison against `A2`.

**Then Task 14.** Whatever wins here on the retention/coverage Pareto plot is the arm to carry
into SWORDS, where alternatives are human-labelled rather than WordNet-generated. Add the
contrastive arm there: `contrastive_train.csv` ships 2,919 human-rejected substitutes, and
`--contrast_beta` with `--negative_column negatives` is already implemented.

**Reporting note.** `A3` at `alpha=0.5` and `A3m` at the matched alpha are two points on an alpha
sweep, not two objectives. Describe them that way; Task 13's "A3 is dominated by A2" conflated a
weaker coefficient with a different method.